# 03 — Visualization Tools

This notebook validates the Plotly visualization layer for the Medical Insight Explorer Agent.

The goal is to convert deterministic analytics outputs into interactive charts that can later be displayed in the Gradio interface.

In [1]:
# Importing libraries and  helpers
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from agent.data_loader import HealthcareDataLoader
from agent.analytics_engine import HealthcareAnalyticsEngine
from agent.visualization_tools import (
    bar_chart_top_values,
    horizontal_bar_chart,
    histogram,
    box_plot,
)

In [2]:
import plotly.io as pio

# Use browser renderer for reliable local Plotly display in classic Jupyter.
# In Gradio/Hugging Face, figures will be returned directly to the UI.
pio.renderers.default = "browser"

In [3]:
# Loading data and initializing engine
loader = HealthcareDataLoader(data_dir=PROJECT_ROOT / "data" / "processed")
tables = loader.load_tables()

engine = HealthcareAnalyticsEngine(tables)

In [4]:
#Chart 1: top inpatient providers
top_inpatient_providers = engine.top_providers_by_claim_count(
    claim_type="inpatient",
    top_n=10,
)

fig = horizontal_bar_chart(
    df=top_inpatient_providers,
    x_col="claim_count",
    y_col="Provider",
    title="Top 10 Inpatient Providers by Claim Count",
    x_label="Claim Count",
    y_label="Provider",
)

fig.show()

In [5]:
#Chart 2: top outpatient providers
top_outpatient_providers = engine.top_providers_by_claim_count(
    claim_type="outpatient",
    top_n=10,
)

fig = horizontal_bar_chart(
    df=top_outpatient_providers,
    x_col="claim_count",
    y_col="Provider",
    title="Top 10 Outpatient Providers by Claim Count",
    x_label="Claim Count",
    y_label="Provider",
)

fig.show()

In [6]:
#Chart 3: inpatient claims by state
inpatient_by_state = engine.claim_distribution_by_state(
    claim_type="inpatient"
).head(10)

fig = bar_chart_top_values(
    df=inpatient_by_state,
    x_col="State",
    y_col="claim_count",
    title="Top 10 States by Inpatient Claim Count",
    x_label="State",
    y_label="Claim Count",
)

fig.show()

In [7]:
#Chart 4: inpatient reimbursement distribution
train_inpatient = tables["train_inpatient"]

fig = histogram(
    df=train_inpatient,
    column="InscClaimAmtReimbursed",
    title="Distribution of Inpatient Claim Reimbursement",
    x_label="Inpatient Claim Amount Reimbursed",
    nbins=40,
)

fig.show()

In [8]:
[col for col in tables["train_beneficiary"].columns if "ChronicCond" in col]

['ChronicCond_Alzheimer',
 'ChronicCond_Heartfailure',
 'ChronicCond_KidneyDisease',
 'ChronicCond_Cancer',
 'ChronicCond_ObstrPulmonary',
 'ChronicCond_Depression',
 'ChronicCond_Diabetes',
 'ChronicCond_IschemicHeart',
 'ChronicCond_Osteoporasis',
 'ChronicCond_rheumatoidarthritis',
 'ChronicCond_stroke']

In [9]:
condition_summary = engine.average_inpatient_cost_by_chronic_condition(
    "ChronicCond_Diabetes"
)

condition_summary

,ChronicCond_Diabetes,count,mean,median,sum
1,2,8012,9002.387668,7000.0,72127130
0,1,32462,8950.338550,7000.0,290545890


In [10]:
fig = bar_chart_top_values(
    df=condition_summary,
    x_col="ChronicCond_Diabetes",
    y_col="mean",
    title="Average Inpatient Reimbursement by Diabetes Indicator",
    x_label="Diabetes Indicator",
    y_label="Average Reimbursement",
)

fig.show()

## Result

The visualization tools successfully convert deterministic analytics outputs into interactive Plotly charts.

These functions can later be reused by the Gradio interface so the agent can return both text summaries and visual outputs.